**Eight notebooks** cover the workflow sequence:
1. [create_datasets_yf.ipynb](01_create_datasets_yf.ipynb) : creates datasets using yfinance
2. [sentiment_feature_building](02_sentiment_feature_building.ipynb): (optional) we calculate the sentiment using finbert on financial news
3. [feature_engineering_yf](03_feature_engineering_yf.ipynb): we compute features from data to later feed into the model
4. [optimizing_xgboost](04_optimizing_xgboost.ipynb): we train a xgboost model to predict returns
5. [evaluate_xgboost](05_evaluate_xgboost.ipynb): we compare the cross-validation performance using various metrics to select the best model. 
6. [model_interpretation](06_model_interpretation.ipynb): we take a closer look at the drivers behind the best model's predictions.
7. `making_out_of_sample_predictions`(this notebook): we predict returns for our out-of-sample period
8. [backtest_backtrader](08_backtest_backtrader.ipynb): evaluate the historical performance of our strategy based on our predictive signals

## Imports & Settings

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
%matplotlib inline

from time import time
import sys, os
from pathlib import Path

import pandas as pd
from scipy.stats import spearmanr

import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)
from utils import MultipleTimeSeriesCV

In [4]:
sns.set_style('whitegrid')

In [5]:
YEAR = 252
idx = pd.IndexSlice

In [6]:
scope_params = ['lookahead', 'train_length', 'test_length']
daily_ic_metrics = ['daily_ic_mean', 'daily_ic_mean_n', 'daily_ic_median', 'daily_ic_median_n']
xgb_train_params = ['learning_rate', 'max_depth', 'n_estimators', 'alpha']

## Generate LightGBM predictions

### Model Configuration

In [7]:
base_params = dict(booster='gbtree',
                   objective='reg:squarederror',
                   verbosity=0)
categoricals = ['year', 'month', 'sector', 'weekday']

In [8]:
lookahead = 1
store = Path('results/xgb_predictions.h5')

### Get Data

In [9]:
data = pd.read_hdf('data_yf.h5', 'model_data').sort_index()

In [10]:
data.index.get_level_values(level=0).unique()

Index(['AAL', 'AAOI', 'AAON', 'AAPL', 'ACGL', 'ACGLO', 'ACHC', 'ACIC', 'ACIW',
       'ACLS',
       ...
       'ZBRA', 'ZD', 'ZEUS', 'ZG', 'ZION', 'ZIONP', 'ZLAB', 'ZS', 'ZVRA',
       'ZYME'],
      dtype='object', name='symbol', length=927)

In [11]:
labels = sorted(data.filter(like='_fwd').columns)
features = data.columns.difference(labels).tolist()
label = f'r{lookahead:02}_fwd'

In [12]:
data = data.loc[idx[:, '2010':], features + [label]].dropna()

In [13]:
for feature in categoricals:
    data[feature] = pd.factorize(data[feature], sort=True)[0]

In [14]:
#xgb_data = xgb.DMatrix(data=data[features], label=data[label], enable_categorical=True)

### Generate predictions

In [15]:
xgb_ic = pd.read_hdf('data/model_tuning.h5', 'xgb/ic')
xgb_daily_ic = pd.read_hdf('data/model_tuning.h5', 'xgb/daily_ic')

In [16]:
def get_xgb_params(data, t=5, best=0):
    param_cols = scope_params[1:] + xgb_train_params + ['boost_rounds']
    df = data[data.lookahead==t].sort_values('ic', ascending=False).iloc[best]
    return df.loc[param_cols]

In [17]:
for position in range(10):
    params = get_xgb_params(xgb_daily_ic,
                            t=lookahead,
                            best=position)

    params = params.to_dict()

    for p in ['max_depth', 'n_estimators','alpha']:
        params[p] = int(params[p])
    train_length = int(params.pop('train_length'))
    test_length = int(params.pop('test_length'))
    num_boost_round = int(params.pop('boost_rounds'))
    params.update(base_params)

    print(f'\nPosition: {position:02}')

    # 1-year out-of-sample period
    n_splits = int(YEAR / test_length)
    cv = MultipleTimeSeriesCV(n_splits=n_splits,
                              test_period_length=test_length,
                              lookahead=lookahead,
                              train_period_length=train_length)

    predictions = []
    start = time()
    for i, (train_idx, test_idx) in enumerate(cv.split(X=data), 1):
        print(i, end=' ', flush=True)
        xgb_train = xgb.DMatrix(
            data=data.iloc[train_idx][features],
            label=data.iloc[train_idx][label],
            enable_categorical=True
        )

        model = xgb.train(params=params,
                        dtrain=xgb_train,
                        num_boost_round=num_boost_round,
                        verbose_eval=False)

        test_set = data.iloc[test_idx, :]
        y_test = test_set.loc[:, label].to_frame('y_test')

        # Create DMatrix for prediction
        xgb_test = xgb.DMatrix(
            data=test_set.loc[:, features],
            enable_categorical=True
        )
        y_pred = model.predict(xgb_test)
        predictions.append(y_test.assign(prediction=y_pred))

    if position == 0:
        test_predictions = (pd.concat(predictions)
                            .rename(columns={'prediction': position}))
    else:
        test_predictions[position] = pd.concat(predictions).prediction

by_day = test_predictions.groupby(level='date')
for position in range(10):
    if position == 0:
        ic_by_day = by_day.apply(lambda x: spearmanr(
            x.y_test, x[position])[0]).to_frame()
    else:
        ic_by_day[position] = by_day.apply(
            lambda x: spearmanr(x.y_test, x[position])[0])
print(ic_by_day.describe())
test_predictions.to_hdf(store, f'xgb/test/{lookahead:02}')


Position: 00
1 2 3 4 5 6 7 8 9 10 11 12 
Position: 01
1 2 3 4 5 6 7 8 9 10 11 12 
Position: 02
1 2 3 4 5 6 7 8 9 10 11 12 
Position: 03
1 2 3 4 5 6 7 8 9 10 11 12 
Position: 04
1 2 3 4 5 6 7 8 9 10 11 12 
Position: 05
1 2 3 4 5 6 7 8 9 10 11 12 
Position: 06
1 2 3 4 
Position: 07
1 2 3 4 
Position: 08
1 2 3 4 
Position: 09
1 2 3 4                 0           1           2           3           4           5  \
count  252.000000  252.000000  252.000000  252.000000  245.000000  245.000000   
mean     0.006395    0.007011    0.004384    0.005911    0.008278    0.008278   
std      0.086233    0.083803    0.083726    0.086457    0.073363    0.073363   
min     -0.273764   -0.220566   -0.233778   -0.229766   -0.221448   -0.221448   
25%     -0.044233   -0.047740   -0.049141   -0.043472   -0.030860   -0.030860   
50%      0.012618    0.013799    0.007175    0.006315    0.007119    0.007119   
75%      0.061547    0.062946    0.053330    0.051577    0.040253    0.040253   
max      0.278495 